# 手写数字识别 — Notebook 入口

这个 notebook 是本项目的**入口**：用 `src/` 里的代码完成训练、评估与预测导出。

## 你将完成什么
1. 安装依赖（只需一次）
2. 在 MNIST 上训练并将产物写入 `outputs/`
3. 评估（混淆矩阵 + 误分类样例图）
4. （可选）对无标签图片文件夹导出预测 CSV

## 你应该重点查看的产物
- `outputs/checkpoints/best_model.pt`
- `outputs/logs/history.json`
- `outputs/logs/run_manifest.json`
- `outputs/figures/training_curves.png`
- `outputs/figures/confusion_matrix.png`
- `outputs/figures/misclassified_grid.png`


## 0) 环境检查
运行下一格代码，确认你在正确的仓库目录与 Python 环境中。

In [ ]:
import sys
from pathlib import Path

print('Python 版本:', sys.version)
print('解释器路径:', sys.executable)
print('当前工作目录:', Path.cwd())
print('是否包含 src/ 目录?:', (Path.cwd() / 'src').is_dir())


## 1) 安装依赖（只需一次）
如果你已经在当前环境安装过依赖，可以跳过。

In [2]:
# 如果在 Jupyter 里安装失败，请改用终端执行：
# python -m pip install -r requirements.txt

import sys
import subprocess

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'])


0

## 2) 训练 MNIST（推荐先跑通基线）
这一部分实际调用的是 `python -m src.train`。

建议先用较小的 epochs（例如 3）验证整条流水线可用，然后再拉长训练做对比实验。

In [ ]:
import sys
import subprocess
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from torchvision import datasets, transforms

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

project_root = Path('.').resolve()
data_root = project_root / 'data'

# 1) 先预览一小部分 MNIST 数据（你能直观看到输入是什么样）
dataset = datasets.MNIST(
    root=str(data_root),
    train=True,
    download=True,
    transform=transforms.ToTensor(),
)

print('MNIST 训练集样本数:', len(dataset))
counts = np.bincount(dataset.targets.numpy(), minlength=10)
print('每个数字的样本数:', {i: int(c) for i, c in enumerate(counts)})

fig, axes = plt.subplots(4, 4, figsize=(6, 6))
for idx, ax in enumerate(axes.flatten()):
    img, label = dataset[idx]
    ax.imshow(img.squeeze(0), cmap='gray')
    ax.set_title(f'标签: {label}')
    ax.axis('off')
fig.suptitle('MNIST 样例（训练集前 16 张）')
plt.tight_layout()
plt.show()

# 2) 调用训练 CLI（会写入 outputs/ 下的 checkpoint / history / 曲线图等）
epochs = 3
batch_size = 64
learning_rate = 1e-3
seed = 42

cmd = [
    sys.executable, '-m', 'src.train',
    '--dataset-name', 'mnist',
    '--project-root', str(project_root),
    '--epochs', str(epochs),
    '--batch-size', str(batch_size),
    '--learning-rate', str(learning_rate),
    '--seed', str(seed),
]

print('运行命令:', ' '.join(cmd))
subprocess.check_call(cmd)


## 3) 检查训练产物
确认训练完成后关键文件都已生成。

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

outputs_dir = Path('outputs')
checkpoint = outputs_dir / 'checkpoints' / 'best_model.pt'
history_path = outputs_dir / 'logs' / 'history.json'
manifest_path = outputs_dir / 'logs' / 'run_manifest.json'

for path in [checkpoint, history_path, manifest_path]:
    print(path, '是否存在?', path.exists())

if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
    print('run_manifest 内容:', manifest)

# 在 notebook 里直接画出训练曲线（不需要手动打开 PNG）
if history_path.exists():
    history = json.loads(history_path.read_text(encoding='utf-8'))

    epochs = range(1, len(history['train_loss']) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))

    axes[0].plot(epochs, history['train_loss'], label='训练')
    axes[0].plot(epochs, history['val_loss'], label='验证')
    axes[0].set_title('损失曲线')
    axes[0].set_xlabel('轮次')
    axes[0].legend()

    axes[1].plot(epochs, history['train_accuracy'], label='训练')
    axes[1].plot(epochs, history['val_accuracy'], label='验证')
    axes[1].set_title('准确率曲线')
    axes[1].set_xlabel('轮次')
    axes[1].legend()

    plt.tight_layout()
    plt.show()

    print('验证集最佳准确率(best_val_accuracy):', history.get('best_val_accuracy'))


## 4) 评估（混淆矩阵 + 误分类样例图）
这一部分调用的是 `python -m src.evaluate`，并将图像产物写入 `outputs/figures/`。

In [6]:
import sys
import subprocess
from pathlib import Path

checkpoint = Path('outputs/checkpoints/best_model.pt').resolve()

cmd = [
    sys.executable, '-m', 'src.evaluate',
    '--checkpoint', str(checkpoint),
    '--dataset-name', 'mnist',
    '--project-root', str(Path('.').resolve()),
]

print('运行命令:', ' '.join(cmd))
subprocess.check_call(cmd)


运行命令: c:\Users\claredz\anaconda3\python.exe -m src.evaluate --checkpoint E:\ALL\学习\AI导论作业-识别手写数字\.worktrees\baseline\outputs\checkpoints\best_model.pt --dataset-name mnist --project-root E:\ALL\学习\AI导论作业-识别手写数字\.worktrees\baseline


0

## 5) 查看图像产物
如果你的 Jupyter 环境支持图片展示，可以在文件浏览器里打开这些 PNG；此处也会检查文件是否存在。

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
from PIL import Image

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

fig_dir = Path('outputs/figures')
figure_paths = {
    '训练曲线': fig_dir / 'training_curves.png',
    '混淆矩阵': fig_dir / 'confusion_matrix.png',
    '误分类样例': fig_dir / 'misclassified_grid.png',
}

for name, path in figure_paths.items():
    print(name, path, '是否存在?', path.exists())

for name, path in figure_paths.items():
    if not path.exists():
        continue
    image = Image.open(path)
    plt.figure(figsize=(8, 4))
    plt.imshow(image)
    plt.title(name)
    plt.axis('off')
    plt.show()


## 6) （可选）对无标签图片导出预测结果
把无标签图片放到一个文件夹里（例如 `sample_predict_images/`），然后运行下一格。

预期输出：`outputs/predictions/predictions.csv`

In [ ]:
import sys
import subprocess
from pathlib import Path

image_dir = Path('sample_predict_images')
if not image_dir.exists():
    raise FileNotFoundError('请创建 sample_predict_images/ 并放入无标签图片后再运行此单元格。')

checkpoint = Path('outputs/checkpoints/best_model.pt').resolve()
cmd = [
    sys.executable, '-m', 'src.predict',
    '--checkpoint', str(checkpoint),
    '--image-dir', str(image_dir.resolve()),
    '--project-root', str(Path('.').resolve()),
]

print('运行命令:', ' '.join(cmd))
subprocess.check_call(cmd)

print('CSV 已生成:', Path('outputs/predictions/predictions.csv').resolve())
